# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is hosted at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m")
print(metadata.description)

## 2. Data Overview
Review available record sets and their details. All entity references use their `@id`.

Let's list all record sets, fields, and columns with their `@id`s.

In [ ]:
# List all available record sets, their IDs, fields, and columns
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets.append(rs['@id'])
        print(f"RecordSet `@id`: {rs['@id']}")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  Field `@id`: {field['@id']}  [name: {field.get('name','')}]" )
                if 'columns' in field:
                    for col in field['columns']:
                        print(f"    Column `@id`: {col['@id']}  [name: {col.get('name','')}]" )
else:
    print('No record sets available in the dataset Croissant metadata.')

## 3. Data Extraction
Let's extract all records from each available record set using its `@id`. Data will be loaded into pandas DataFrames. If you need to extract a specific record set, use its `@id` as shown in the overview step above.

In [ ]:
dataframes = {}

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        recset_id = rs['@id']
        print(f"Extracting records from RecordSet: {recset_id}")
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
    # For demonstration, let's print columns and head for the first record set
    first_rs_id = metadata.record_sets[0]['@id']
    print(f"\nColumns in {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print('No record sets found, so no data extracted.')

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA on the extracted data.

- Filter records by a numeric field.
- Normalize the numeric field.
- Group or aggregate by a categorical field.

_Replace the variable values below with actual field `@id`s based on your data overview._

In [ ]:
# Example EDA on the first record set, using @id references
if dataframes:
    first_rs_id = next(iter(dataframes.keys()))
    df = dataframes[first_rs_id]
    print(f"Exploring RecordSet: {first_rs_id}")
    print(f"Fields: {df.columns.tolist()}")

    # Guess a numeric field if any (using heuristics for demonstration)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        print(f"Numeric field selected for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
        # Filter records
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} (z-score):")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Guess a grouping field (categorical)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No extracted data available for EDA.')

## 5. Visualization
Visualize the distribution of the numeric field and relationship with the grouping field.

_If no numeric or grouping field is found in your record set, you may need to examine the fields using the Data Overview step and adjust the field `@id` accordingly._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable fields for visualization found.')

## 6. Conclusion
- This notebook demonstrated the use of the `mlcroissant` library to load and explore the FAIR2 dataset.
- We accessed data via record set, field, and column `@id`s, in compliance with the Croissant schema structure.
- Basic exploratory analysis and visualization steps were performed.

For more advanced insights, consider merging or joining multiple record sets by their reference keys, or applying custom feature engineering using the rich metadata schema provided.